# Whakaari Dataset Construction

This notebook builds the Whakaari case-study dataset for Cause–Trigger analysis. It combines waveform-derived hydrothermal/seismic features with weather, gas, and deformation variables on a common hourly grid.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from whakaari_config import (
    WHAKAARI_START,
    WHAKAARI_END,
    WHAKAARI_ERUPTION_TIME,
    TILDE_SUMMARY_URL,
    TILDE_DATA_URL,
    WHAKAARI_LAT,
    WHAKAARI_LON,
    WHAKAARI_WAVEFORM_CONFIG,
)

from whakaari_geonet import (
    stations_for,
    filter_whakaari,
    load_so2_flux,
    load_gnss_deformation,
    load_weather_vars,
)

from whakaari_waveform import build_waveform_dataset

from whakaari_dataset import (
    build_master_dataframe,
    prepare_analysis_dataframe,
    scale_analysis_dataframe,
    save_whakaari_datasets,
)

from whakaari_plotting_utils import (
    dataset_health_report,
    plot_whakaari_waveforms,
    plot_whakaari_external,
    plot_whakaari_scaled,
    plot_with_eruption_time,
)

client = Client("GEONET")

## 1. GeoNet and external data

We retrieve SO₂ flux and GNSS deformation from GeoNet/Tilde, and hourly weather variables from Open-Meteo. API and pressure drop are derived as hydrothermal forcing proxies.

In [ ]:
so2_st = filter_whakaari(
    stations_for(
        TILDE_SUMMARY_URL,
        "scandoas",
        name="gasflux",
        method="reviewed",
        aspect="SO2",
    )
)

so2_st

In [ ]:
so2 = load_so2_flux(TILDE_DATA_URL, WHAKAARI_START, WHAKAARI_END)
gnss = load_gnss_deformation(TILDE_DATA_URL, WHAKAARI_START, WHAKAARI_END)
weather_vars = load_weather_vars(
    WHAKAARI_LAT,
    WHAKAARI_LON,
    WHAKAARI_START,
    WHAKAARI_END,
)

## 2. Waveform feature extraction

Waveform data from WSRZ are processed into hourly seismic features: hydrothermal tremor RMS, spectral ratio, HF event rate, and a continuous tremor-response energy variable.

In [ ]:
waveform_df, waveform_failures = build_waveform_dataset(
    client=client,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    cfg=WHAKAARI_WAVEFORM_CONFIG,
)

## 3. Final hourly dataset

All variables are aligned to a common hourly timeline. Missing values are handled according to variable type, and the final dataset is saved in raw and scaled forms.

In [ ]:
master_df = build_master_dataframe(
    wave=waveform_df,
    weather_vars=weather_vars,
    so2=so2,
    gnss=gnss,
    start=WHAKAARI_START,
    end=WHAKAARI_END,
    master_freq="1h",
)

analysis_df = prepare_analysis_dataframe(master_df)

analysis_scaled, analysis_prepped, scaler = scale_analysis_dataframe(
    analysis_df
)

save_whakaari_datasets(
    master_df=master_df,
    analysis_df=analysis_df,
    analysis_scaled=analysis_scaled,
    output_dir="../whakaari_data",
)

## 4. Dataset checks

We inspect missingness, summary statistics, and timestamp consistency before using the dataset for causal discovery.

In [ ]:
display(dataset_health_report(master_df, "Whakaari master_df"))
display(dataset_health_report(analysis_df, "Whakaari analysis_df"))
display(dataset_health_report(analysis_scaled, "Whakaari analysis_scaled"))

## 5. Visual inspection

The final variables are plotted with the eruption time marked. This helps assess whether the causal-analysis window and variables are physically meaningful.

In [ ]:
plot_whakaari_waveforms(analysis_df)
plot_whakaari_external(analysis_df)
plot_whakaari_scaled(analysis_scaled)

In [ ]:
plot_with_eruption_time(
    analysis_df,
    [
        "hydro_rms_2_5",
        "ratio_4p5_8_over_8_16",
        "hf_event_rate_2_5",
        "SO2_flux",
        "GNSS_deformation",
        "API",
        "pressure_drop",
        "effect_tremor_rms_5_15",
    ],
    WHAKAARI_ERUPTION_TIME,
)